# 01. 메이플 인벤 CSV 입력

원본 CSV의 경로와 필수 컬럼을 확인하고, 다음 단계가 읽을 원본 스냅샷을 저장합니다. 추가 CSV가 생기면 `CSV_PATTERNS`에 경로나 glob을 추가하면 됩니다.

In [1]:
from pathlib import Path
import csv
import glob
import json
import os
import tempfile

import pandas as pd
from IPython.display import display

CSV_NAME = 'maple_inven_rag_원본(1~10p).csv'
MODEL_NAME = 'jhgan/ko-sroberta-multitask'
CHUNK_TOKENS = 100
OVERLAP_TOKENS = 20
MAX_TOKENS = 128
BATCH_SIZE = 32

def find_data_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / 'ㅋㅌㅊ',
        Path.cwd().parent,
        Path.cwd().parent / 'ㅋㅌㅊ',
    ]
    checked = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        if (resolved / CSV_NAME).is_file():
            return resolved
    raise FileNotFoundError(f'{CSV_NAME}이 있는 ㅋㅌㅊ 폴더를 찾지 못했습니다.')

DATA_DIR = find_data_dir()
OUTPUT_ROOT = DATA_DIR / 'output'
INTERMEDIATE_DIR = OUTPUT_ROOT / 'intermediate'
RAW_PATH = INTERMEDIATE_DIR / 'maple_inven_tips_raw.json'
SETTINGS_PATH = INTERMEDIATE_DIR / 'pipeline_settings.json'
CSV_PATTERNS = [str(DATA_DIR / CSV_NAME)]

print('데이터 폴더:', DATA_DIR)
print('입력 패턴:', CSV_PATTERNS)

데이터 폴더: C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3\ㅋㅌㅊ
입력 패턴: ['C:\\Users\\Playdata\\Desktop\\team3_ 프로젝트1\\mle-01-p1-team3\\ㅋㅌㅊ\\maple_inven_rag_원본(1~10p).csv']


In [2]:
REQUIRED_COLUMNS = frozenset({'url', 'title', 'content'})
OPTIONAL_COLUMNS = ('category', 'author', 'created_at', 'views', 'likes')

def atomic_write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(
            'w', encoding='utf-8', newline='', dir=path.parent, delete=False
        ) as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2)
            stream.write('\n')
            temporary = Path(stream.name)
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

def discover_input_files(patterns):
    discovered = []
    seen = set()
    for raw_pattern in patterns:
        pattern = str(raw_pattern)
        literal = Path(pattern)
        matches = [literal] if literal.is_file() else [Path(v) for v in sorted(glob.glob(pattern))]
        if not matches:
            raise FileNotFoundError(f'입력 CSV를 찾지 못했습니다: {pattern}')
        for match in matches:
            resolved = match.resolve()
            if resolved.suffix.lower() != '.csv':
                raise ValueError(f'CSV 파일만 입력할 수 있습니다: {resolved}')
            if resolved not in seen:
                seen.add(resolved)
                discovered.append(resolved)
    return discovered

def load_csv_rows(paths):
    rows = []
    for path in paths:
        with path.open('r', encoding='utf-8-sig', newline='') as stream:
            reader = csv.DictReader(stream)
            columns = set(reader.fieldnames or ())
            missing = sorted(REQUIRED_COLUMNS - columns)
            if missing:
                raise ValueError(f"{path} 필수 컬럼 누락: {', '.join(missing)}")
            for row_number, row in enumerate(reader, start=2):
                normalized = {key: value or '' for key, value in row.items() if key is not None}
                for name in OPTIONAL_COLUMNS:
                    normalized.setdefault(name, '')
                normalized['__source_file'] = str(path.resolve())
                normalized['__source_row'] = row_number
                rows.append(normalized)
    return rows

In [3]:
input_files = discover_input_files(CSV_PATTERNS)
raw_rows = load_csv_rows(input_files)
settings = {
    'input_files': [str(path) for path in input_files],
    'model_name': MODEL_NAME,
    'chunk_tokens': CHUNK_TOKENS,
    'overlap_tokens': OVERLAP_TOKENS,
    'max_tokens': MAX_TOKENS,
    'batch_size': BATCH_SIZE,
}
atomic_write_json(RAW_PATH, raw_rows)
atomic_write_json(SETTINGS_PATH, settings)

summary = {
    '입력 파일 수': len(input_files),
    '입력 행 수': len(raw_rows),
    '필수 컬럼': sorted(REQUIRED_COLUMNS),
    '중간 저장 파일': str(RAW_PATH),
}
display(summary)
preview_columns = ['category', 'title', 'created_at', 'views', 'likes', 'url']
display(pd.DataFrame(raw_rows)[preview_columns].head(5))

{'입력 파일 수': 1,
 '입력 행 수': 300,
 '필수 컬럼': ['content', 'title', 'url'],
 '중간 저장 파일': 'C:\\Users\\Playdata\\Desktop\\team3_ 프로젝트1\\mle-01-p1-team3\\ㅋㅌㅊ\\output\\intermediate\\maple_inven_tips_raw.json'}

,category,title,created_at,views,likes,url
0,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,2026-08-03 12:43,12988,18,https://www.inven.co.kr/board/maple/2304/48082
1,메이플M,메이플M 렌 250 육성 이벤트 공략 (무과금),2026-08-01 15:52,37151,12,https://www.inven.co.kr/board/maple/2304/48066
2,기타,"[울티마 스쿼드] 스테이지 권장레벨, 잠재옵션표, 스킬퍼뎀, 장비 리스트 및 능력치 공유",2026-07-26 11:56,318931,29,https://www.inven.co.kr/board/maple/2304/48012
3,아이템,울티마 스쿼드 장비 / 잠재 정보,2026-07-24 14:21,96343,13,https://www.inven.co.kr/board/maple/2304/47984
4,사냥,울티마 스쿼드 정보들 (테섭 기준),2026-07-23 07:41,339767,37,https://www.inven.co.kr/board/maple/2304/47971
